## Preliminaries

In [1]:
from rdkit.Chem import Descriptors
import pandas as pd
import os
import sys
import random
import numpy as np
import torch

In [2]:

#set random seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

In [3]:
import logging
logging.basicConfig()
root_logger = logging.getLogger()
root_logger.setLevel(logging.WARN)

#### Import libraries

In [4]:
from ccrlib import *
from ccrlib.fragmentation import *
logger.setLevel(logging.WARN)

#### Relevant parameters

In [5]:
cut_type="synthesizable" # or "single" or "recap" or "fg"
max_cuts = 5
min_rel_core_size=0.666
max_frag_size = 13
max_time=100
mol_filter = lambda x: Descriptors.MolWt(x)<=1000
mode = "ccr"

#### MY DATA

In [6]:
import pickle
with open('../datasets/df_random_vs_st_dataset_revised.pkl', 'rb') as f:
    data = pickle.load(f)

In [7]:
data

,nonstereo_aromatic_smiles,accession,chembl_cid,chembl_tid,label
0,Brc1ccc(-c2cnc3nnc(Cc4c[nH]c5ccccc45)n3n2)cc1,P08581,CHEMBL3799231,CHEMBL3717,1
1,Brc1ccc(-c2cnc3nnc(Cc4ccc5ncccc5c4)n3n2)cc1,P08581,CHEMBL3798130,CHEMBL3717,1
2,Brc1ccc(-c2nc3ccc(Nc4ncnc5ccccc45)cc3[nH]2)cc1,P08581,CHEMBL3321905,CHEMBL3717,1
3,C#CCC(Oc1c(N)ncc2c(-c3cnn(C4CCNCC4)c3)coc12)c1...,P08581,CHEMBL2401818,CHEMBL3717,1
4,C#Cc1c(Oc2ccc(-n3cc(C(=O)NCc4ccc(F)cc4)ccc3=O)...,P08581,CHEMBL3309958,CHEMBL3717,1
...,...,...,...,...,...
43069,CN(CCC(=O)N1CCN(c2ccc(F)c(F)c2)CC1)CCC1c2ccccc...,P27487,CHEMBL473763,CHEMBL1917,0
43070,COc1ccc(COc2ccc(C3CN(C(=O)c4cc(OCCO)ccn4)C3)cc...,P27487,CHEMBL4543929,CHEMBL1844,0
43071,NC(CCC(=O)NC(CSC(=O)N(O)c1ccc(Cl)cc1)C(=O)NCC(...,P27487,CHEMBL131578,CHEMBL2424,0
43072,CC1CCc2ccccc2N1C(=O)c1ccc2c(c1)NC(=O)CO2,P27487,CHEMBL4474881,CHEMBL1994,0


In [8]:
targets = data['accession'].unique()

filtered_data = data[data['accession'] == 'Q16790']
filtered_data
    

,nonstereo_aromatic_smiles,accession,chembl_cid,chembl_tid,label
7874,Brc1ccc(-c2nn(-c3ccccc3)cc2-c2nc3ccccc3[nH]2)cc1,Q16790,CHEMBL3809848,CHEMBL3594,1
7875,C#CCCCOc1ccc2ccc(=O)oc2c1,Q16790,CHEMBL3121782,CHEMBL3594,1
7876,C#CCCCOc1ccc2ccc(=S)oc2c1,Q16790,CHEMBL3760055,CHEMBL3594,1
7877,C#CCN(CC#C)CCCCOc1ccc(S(N)(=O)=O)cc1,Q16790,CHEMBL4435346,CHEMBL3594,1
7878,C#CCN(CC#C)CCc1ccc(S(N)(=O)=O)cc1,Q16790,CHEMBL4566851,CHEMBL3594,1
...,...,...,...,...,...
15427,CN1CCc2nc(C(=O)NC3CC(C(=O)O)CCC3NC(=O)c3cc4cc(...,Q16790,CHEMBL503078_CHEMBL538657,CHEMBL244,0
15428,Nc1nc(Oc2ccccc2)nc2nc(-c3ccco3)nn12,Q16790,CHEMBL1093313,CHEMBL251,0
15429,Cc1noc2c1-c1sc(C)c(C)c1C(N1CCOCC1)=NC2CC(N)=O,Q16790,CHEMBL2431086,CHEMBL1163125,0
15430,O=C(NO)c1ccc(CN2C(=O)CCC2=O)cc1-c1ccccc1,Q16790,CHEMBL4776962,CHEMBL2716,0


In [9]:
#keeping only active

filtered_data = filtered_data[filtered_data['label'] == 1]
filtered_data
    

,nonstereo_aromatic_smiles,accession,chembl_cid,chembl_tid,label
7874,Brc1ccc(-c2nn(-c3ccccc3)cc2-c2nc3ccccc3[nH]2)cc1,Q16790,CHEMBL3809848,CHEMBL3594,1
7875,C#CCCCOc1ccc2ccc(=O)oc2c1,Q16790,CHEMBL3121782,CHEMBL3594,1
7876,C#CCCCOc1ccc2ccc(=S)oc2c1,Q16790,CHEMBL3760055,CHEMBL3594,1
7877,C#CCN(CC#C)CCCCOc1ccc(S(N)(=O)=O)cc1,Q16790,CHEMBL4435346,CHEMBL3594,1
7878,C#CCN(CC#C)CCc1ccc(S(N)(=O)=O)cc1,Q16790,CHEMBL4566851,CHEMBL3594,1
...,...,...,...,...,...
11648,S=C(S)OCc1ccccc1,Q16790,CHEMBL2380749,CHEMBL3594,1
11649,S=c1ccc2ccc(OCc3cn(-c4ccccc4)nn3)cc2o1,Q16790,CHEMBL3758827,CHEMBL3594,1
11650,S=c1ccc2ccccc2o1,Q16790,CHEMBL3759409,CHEMBL3594,1
11651,S=c1cccco1,Q16790,CHEMBL1934664,CHEMBL3594,1


In [10]:
filtered_data["smiles"] = filtered_data["nonstereo_aromatic_smiles"]

filtered_data["cid_tid_smiles"] = (
    filtered_data["chembl_cid"] + "&" + filtered_data["chembl_tid"] + "&" + filtered_data["smiles"]
)

# Select only required columns
df_new = filtered_data[["smiles", "cid_tid_smiles"]]

# Save if needed
df_new.to_csv("Q16790.csv", index=False)

/tmp/ipykernel_1050166/1454509603.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data["smiles"] = filtered_data["nonstereo_aromatic_smiles"]
/tmp/ipykernel_1050166/1454509603.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data["cid_tid_smiles"] = (


In [11]:
pd.read_csv("Q16790.csv")

,smiles,cid_tid_smiles
0,Brc1ccc(-c2nn(-c3ccccc3)cc2-c2nc3ccccc3[nH]2)cc1,CHEMBL3809848&CHEMBL3594&Brc1ccc(-c2nn(-c3cccc...
1,C#CCCCOc1ccc2ccc(=O)oc2c1,CHEMBL3121782&CHEMBL3594&C#CCCCOc1ccc2ccc(=O)o...
2,C#CCCCOc1ccc2ccc(=S)oc2c1,CHEMBL3760055&CHEMBL3594&C#CCCCOc1ccc2ccc(=S)o...
3,C#CCN(CC#C)CCCCOc1ccc(S(N)(=O)=O)cc1,CHEMBL4435346&CHEMBL3594&C#CCN(CC#C)CCCCOc1ccc...
4,C#CCN(CC#C)CCc1ccc(S(N)(=O)=O)cc1,CHEMBL4566851&CHEMBL3594&C#CCN(CC#C)CCc1ccc(S(...
...,...,...
3774,S=C(S)OCc1ccccc1,CHEMBL2380749&CHEMBL3594&S=C(S)OCc1ccccc1
3775,S=c1ccc2ccc(OCc3cn(-c4ccccc4)nn3)cc2o1,CHEMBL3758827&CHEMBL3594&S=c1ccc2ccc(OCc3cn(-c...
3776,S=c1ccc2ccccc2o1,CHEMBL3759409&CHEMBL3594&S=c1ccc2ccccc2o1
3777,S=c1cccco1,CHEMBL1934664&CHEMBL3594&S=c1cccco1


## Generate CCRs

## Function call combining all steps (preferred)

In [12]:
from ccrlib.ccr import run_ccr, verify_series

In [13]:
import logging
logging.getLogger("ccr.ccr").setLevel(logging.ERROR)

In [14]:
df_ccr_results = pd.DataFrame()
results_list = []
tids_wo_ccr = []

for tid in ['CHEMBL267']:

    suppl = Chem.SmilesMolSupplier('Q16790.csv', delimiter=',', smilesColumn=0, nameColumn=1)

    try:
        result = run_ccr(suppl, cut_type, max_cuts, min_rel_core_size, max_frag_size, max_time, mol_filter, basename= None)
        # Verify the correctness of fragmentations
        verify_s_unique = "Verification "+("okay" if verify_series(result.unique,result.smiles) else "failed")
        verify_s = "Verification "+("okay" if verify_series(result.all_series,result.smiles) else "failed")
        
        if verify_s_unique == "Verification okay" and verify_s == "Verification okay":
            
            df = result.unique.to_dataframe()
            
            df_core = df.copy()[['name', 'core', 'substituents']]
            df_core['cid'] = df_core['name'].apply(lambda x: x.split('&')[0])
            df_core['tid'] = df_core['name'].apply(lambda x: x.split('&')[1])
            df_core['smiles'] = df_core['name'].apply(lambda x: x.split('&')[2])
            df_core.drop(columns=['name'], inplace=True)
            df_ccr_results = pd.concat([df_ccr_results, df_core], axis=0)
            
            results_list.append(result)
        else:
            print(f"Verification for {tid} failed: {verify_s_unique}, {verify_s}")  
    except Exception as e:
        print(f"Error processing : {e}")
        tids_wo_ccr.append(tid)


# smiles: 3779
# duplicates: 0
# discarded molecules: 0
# frames: 10021
# cuts:  12511
Time:  6.898935794830322
FPS:  1813.4681017578168
# frames: 13800
# cuts:  16290
# hcores:  13124
# cores in hcores:  13800
#hcores:  13124 #cpds:  16290 #unique cpds: 3779
Removed single-cpd series:
#hcores:  1329 #cpds:  4495 #unique cpds: 1791
Removed sub-series:
#hcores:  572 #cpds:  2166 #unique cpds: 1791
Optimized R-sites
#hcores:  572 #cpds:  2166 #unique cpds: 1791
Unique assignments:
#hcores:  432 #cpds:  1720 #unique cpds: 1720
Optimized R-sites for unique ccrs
#hcores:  432 #cpds:  1720 #unique cpds: 1720


Failed to find the pandas get_adjustment() function to patch
Failed to patch pandas - PandasTools will have limited functionality
Failed to patch pandas - unable to change molecule rendering
Failed to patch pandas - unable to change molecule rendering


In [15]:

df_ccr_results.reset_index(drop=True, inplace=True)
df_ccr_results

,core,substituents,cid,tid,smiles
0,c1ccc(-n2cc(-c3nc4ccccc4[nH]3)c([*:1])n2)cc1,Brc1ccc([*:1])cc1,CHEMBL3809848,CHEMBL3594,Brc1ccc(-c2nn(-c3ccccc3)cc2-c2nc3ccccc3[nH]2)cc1
1,c1ccc(-n2cc(-c3nc4ccccc4[nH]3)c([*:1])n2)cc1,COc1ccc([*:1])cc1,CHEMBL3613544,CHEMBL3594,COc1ccc(-c2nn(-c3ccccc3)cc2-c2nc3ccccc3[nH]2)cc1
2,c1ccc(-n2cc(-c3nc4ccccc4[nH]3)c([*:1])n2)cc1,Cc1ccc([*:1])cc1,CHEMBL3809267,CHEMBL3594,Cc1ccc(-c2nn(-c3ccccc3)cc2-c2nc3ccccc3[nH]2)cc1
3,c1ccc(-n2cc(-c3nc4ccccc4[nH]3)c([*:1])n2)cc1,Clc1ccc([*:1])cc1,CHEMBL3613534,CHEMBL3594,Clc1ccc(-c2nn(-c3ccccc3)cc2-c2nc3ccccc3[nH]2)cc1
4,c1ccc(-n2cc(-c3nc4ccccc4[nH]3)c([*:1])n2)cc1,Fc1ccc([*:1])cc1,CHEMBL3613524,CHEMBL3594,Fc1ccc(-c2nn(-c3ccccc3)cc2-c2nc3ccccc3[nH]2)cc1
...,...,...,...,...,...
1715,O=P(O)(O)C(Nc1cccc([*:1])c1)P(=O)(O)O,[*:1],CHEMBL2440580,CHEMBL3594,O=P(O)(O)C(Nc1ccccc1)P(=O)(O)O
1716,O=P(O)(O)C(Nc1cccc([*:1])c1)P(=O)(O)O,c1ccc([*:1])cc1,CHEMBL55371,CHEMBL3594,O=P(O)(O)C(Nc1cccc(-c2ccccc2)c1)P(=O)(O)O
1717,O=c1[nH]c2cc(Cl)c([*:1])cc2c(=O)n1O,[*:1],CHEMBL80594,CHEMBL3594,O=c1[nH]c2cc(Cl)ccc2c(=O)n1O
1718,O=c1[nH]c2cc(Cl)c([*:1])cc2c(=O)n1O,c1ccn([*:1])c1,CHEMBL385438,CHEMBL3594,O=c1[nH]c2cc(Cl)c(-n3cccc3)cc2c(=O)n1O


In [16]:
core_id = {core: idx for idx, core in enumerate(df_ccr_results['core'].unique())}
df_ccr_results['analogue_series_id'] = df_ccr_results['core'].map(core_id)
df_ccr_results

,core,substituents,cid,tid,smiles,analogue_series_id
0,c1ccc(-n2cc(-c3nc4ccccc4[nH]3)c([*:1])n2)cc1,Brc1ccc([*:1])cc1,CHEMBL3809848,CHEMBL3594,Brc1ccc(-c2nn(-c3ccccc3)cc2-c2nc3ccccc3[nH]2)cc1,0
1,c1ccc(-n2cc(-c3nc4ccccc4[nH]3)c([*:1])n2)cc1,COc1ccc([*:1])cc1,CHEMBL3613544,CHEMBL3594,COc1ccc(-c2nn(-c3ccccc3)cc2-c2nc3ccccc3[nH]2)cc1,0
2,c1ccc(-n2cc(-c3nc4ccccc4[nH]3)c([*:1])n2)cc1,Cc1ccc([*:1])cc1,CHEMBL3809267,CHEMBL3594,Cc1ccc(-c2nn(-c3ccccc3)cc2-c2nc3ccccc3[nH]2)cc1,0
3,c1ccc(-n2cc(-c3nc4ccccc4[nH]3)c([*:1])n2)cc1,Clc1ccc([*:1])cc1,CHEMBL3613534,CHEMBL3594,Clc1ccc(-c2nn(-c3ccccc3)cc2-c2nc3ccccc3[nH]2)cc1,0
4,c1ccc(-n2cc(-c3nc4ccccc4[nH]3)c([*:1])n2)cc1,Fc1ccc([*:1])cc1,CHEMBL3613524,CHEMBL3594,Fc1ccc(-c2nn(-c3ccccc3)cc2-c2nc3ccccc3[nH]2)cc1,0
...,...,...,...,...,...,...
1715,O=P(O)(O)C(Nc1cccc([*:1])c1)P(=O)(O)O,[*:1],CHEMBL2440580,CHEMBL3594,O=P(O)(O)C(Nc1ccccc1)P(=O)(O)O,430
1716,O=P(O)(O)C(Nc1cccc([*:1])c1)P(=O)(O)O,c1ccc([*:1])cc1,CHEMBL55371,CHEMBL3594,O=P(O)(O)C(Nc1cccc(-c2ccccc2)c1)P(=O)(O)O,430
1717,O=c1[nH]c2cc(Cl)c([*:1])cc2c(=O)n1O,[*:1],CHEMBL80594,CHEMBL3594,O=c1[nH]c2cc(Cl)ccc2c(=O)n1O,431
1718,O=c1[nH]c2cc(Cl)c([*:1])cc2c(=O)n1O,c1ccn([*:1])c1,CHEMBL385438,CHEMBL3594,O=c1[nH]c2cc(Cl)c(-n3cccc3)cc2c(=O)n1O,431


In [17]:
df_ccr_results.to_csv(os.path.join('', 'df_ccr_results_Q16790.csv'), index=False)

In [18]:
pd.read_csv('df_ccr_results_Q16790.csv')

,core,substituents,cid,tid,smiles,analogue_series_id
0,c1ccc(-n2cc(-c3nc4ccccc4[nH]3)c([*:1])n2)cc1,Brc1ccc([*:1])cc1,CHEMBL3809848,CHEMBL3594,Brc1ccc(-c2nn(-c3ccccc3)cc2-c2nc3ccccc3[nH]2)cc1,0
1,c1ccc(-n2cc(-c3nc4ccccc4[nH]3)c([*:1])n2)cc1,COc1ccc([*:1])cc1,CHEMBL3613544,CHEMBL3594,COc1ccc(-c2nn(-c3ccccc3)cc2-c2nc3ccccc3[nH]2)cc1,0
2,c1ccc(-n2cc(-c3nc4ccccc4[nH]3)c([*:1])n2)cc1,Cc1ccc([*:1])cc1,CHEMBL3809267,CHEMBL3594,Cc1ccc(-c2nn(-c3ccccc3)cc2-c2nc3ccccc3[nH]2)cc1,0
3,c1ccc(-n2cc(-c3nc4ccccc4[nH]3)c([*:1])n2)cc1,Clc1ccc([*:1])cc1,CHEMBL3613534,CHEMBL3594,Clc1ccc(-c2nn(-c3ccccc3)cc2-c2nc3ccccc3[nH]2)cc1,0
4,c1ccc(-n2cc(-c3nc4ccccc4[nH]3)c([*:1])n2)cc1,Fc1ccc([*:1])cc1,CHEMBL3613524,CHEMBL3594,Fc1ccc(-c2nn(-c3ccccc3)cc2-c2nc3ccccc3[nH]2)cc1,0
...,...,...,...,...,...,...
1715,O=P(O)(O)C(Nc1cccc([*:1])c1)P(=O)(O)O,[*:1],CHEMBL2440580,CHEMBL3594,O=P(O)(O)C(Nc1ccccc1)P(=O)(O)O,430
1716,O=P(O)(O)C(Nc1cccc([*:1])c1)P(=O)(O)O,c1ccc([*:1])cc1,CHEMBL55371,CHEMBL3594,O=P(O)(O)C(Nc1cccc(-c2ccccc2)c1)P(=O)(O)O,430
1717,O=c1[nH]c2cc(Cl)c([*:1])cc2c(=O)n1O,[*:1],CHEMBL80594,CHEMBL3594,O=c1[nH]c2cc(Cl)ccc2c(=O)n1O,431
1718,O=c1[nH]c2cc(Cl)c([*:1])cc2c(=O)n1O,c1ccn([*:1])c1,CHEMBL385438,CHEMBL3594,O=c1[nH]c2cc(Cl)c(-n3cccc3)cc2c(=O)n1O,431


In [22]:
for target in targets:
    filtered_data = data[data['accession'] == target]
    #keep only active:
    filtered_data = filtered_data[filtered_data['label'] == 1]

    filtered_data["smiles"] = filtered_data["nonstereo_aromatic_smiles"]

    filtered_data["cid_tid_smiles"] = (
        filtered_data["chembl_cid"] + "&" + filtered_data["chembl_tid"] + "&" + filtered_data["smiles"]
    )

    # Select only required columns
    df_new = filtered_data[["smiles", "cid_tid_smiles"]]

    # Save if needed
    df_new.to_csv(f"{target}.csv", index=False)

    df_ccr_results = pd.DataFrame()
    results_list = []
    tids_wo_ccr = []

    suppl = Chem.SmilesMolSupplier(f'{target}.csv', delimiter=',', smilesColumn=0, nameColumn=1)

    try:
        result = run_ccr(suppl, cut_type, max_cuts, min_rel_core_size, max_frag_size, max_time, mol_filter, basename= None)
        # Verify the correctness of fragmentations
        verify_s_unique = "Verification "+("okay" if verify_series(result.unique,result.smiles) else "failed")
        verify_s = "Verification "+("okay" if verify_series(result.all_series,result.smiles) else "failed")
        
        if verify_s_unique == "Verification okay" and verify_s == "Verification okay":
            
            df = result.unique.to_dataframe()
            
            df_core = df.copy()[['name', 'core', 'substituents']]
            df_core['cid'] = df_core['name'].apply(lambda x: x.split('&')[0])
            df_core['tid'] = df_core['name'].apply(lambda x: x.split('&')[1])
            df_core['smiles'] = df_core['name'].apply(lambda x: x.split('&')[2])
            df_core.drop(columns=['name'], inplace=True)
            df_ccr_results = pd.concat([df_ccr_results, df_core], axis=0)
            
            results_list.append(result)
        else:
            print(f"Verification for {tid} failed: {verify_s_unique}, {verify_s}")  
    except Exception as e:
        print(f"Error processing : {e}")
        tids_wo_ccr.append(tid)

    df_ccr_results.reset_index(drop=True, inplace=True)
    core_id = {core: idx for idx, core in enumerate(df_ccr_results['core'].unique())}
    df_ccr_results['analogue_series_id'] = df_ccr_results['core'].map(core_id)


    #save
    df_ccr_results.to_csv(os.path.join('', f'df_ccr_results_{target}.csv'), index=False)




    

# smiles: 1414
# duplicates: 0
# discarded molecules: 0
# frames: 3456
# cuts:  4905
Time:  3.4979259967803955
FPS:  1402.2595116405325
# frames: 4870
# cuts:  6319
# hcores:  4669
# cores in hcores:  4870
#hcores:  4669 #cpds:  6319 #unique cpds: 1414
Removed single-cpd series:
#hcores:  591 #cpds:  2241 #unique cpds: 965
Removed sub-series:
#hcores:  254 #cpds:  1155 #unique cpds: 965
Optimized R-sites
#hcores:  254 #cpds:  1155 #unique cpds: 965
Unique assignments:
#hcores:  192 #cpds:  931 #unique cpds: 931
Optimized R-sites for unique ccrs
#hcores:  192 #cpds:  931 #unique cpds: 931


Failed to patch pandas - unable to change molecule rendering
Failed to patch pandas - unable to change molecule rendering


# smiles: 2523
# duplicates: 0
# discarded molecules: 0
# frames: 8599
# cuts:  12135
Time:  8.322160959243774
FPS:  1458.1549262780293
# frames: 11122
# cuts:  14658
# hcores:  10679
# cores in hcores:  11122
#hcores:  10679 #cpds:  14658 #unique cpds: 2523
Removed single-cpd series:
#hcores:  1592 #cpds:  5571 #unique cpds: 1668
Removed sub-series:
#hcores:  496 #cpds:  2077 #unique cpds: 1668
Optimized R-sites
#hcores:  496 #cpds:  2077 #unique cpds: 1668
Unique assignments:
#hcores:  368 #cpds:  1621 #unique cpds: 1621
Optimized R-sites for unique ccrs
#hcores:  368 #cpds:  1621 #unique cpds: 1621


Failed to patch pandas - unable to change molecule rendering
Failed to patch pandas - unable to change molecule rendering


# smiles: 3779
# duplicates: 0
# discarded molecules: 0
# frames: 10021
# cuts:  12511
Time:  7.839963436126709
FPS:  1595.7982587455779
# frames: 13800
# cuts:  16290
# hcores:  13124
# cores in hcores:  13800
#hcores:  13124 #cpds:  16290 #unique cpds: 3779
Removed single-cpd series:
#hcores:  1329 #cpds:  4495 #unique cpds: 1791
Removed sub-series:
#hcores:  572 #cpds:  2166 #unique cpds: 1791
Optimized R-sites
#hcores:  572 #cpds:  2166 #unique cpds: 1791
Unique assignments:
#hcores:  432 #cpds:  1720 #unique cpds: 1720
Optimized R-sites for unique ccrs
#hcores:  432 #cpds:  1720 #unique cpds: 1720


...

Failed to patch pandas - unable to change molecule rendering
Failed to patch pandas - unable to change molecule rendering


# smiles: 2028
# duplicates: 0
# discarded molecules: 0
# frames: 12065
# cuts:  14590
Time:  11.202566146850586
FPS:  1302.3801697526003
# frames: 14093
# cuts:  16618
# hcores:  13685
# cores in hcores:  14093
#hcores:  13685 #cpds:  16618 #unique cpds: 2028
Removed single-cpd series:
#hcores:  1615 #cpds:  4548 #unique cpds: 1307
Removed sub-series:
#hcores:  528 #cpds:  1710 #unique cpds: 1307
Optimized R-sites
#hcores:  528 #cpds:  1710 #unique cpds: 1307
Unique assignments:
#hcores:  357 #cpds:  1242 #unique cpds: 1242
Optimized R-sites for unique ccrs
#hcores:  357 #cpds:  1242 #unique cpds: 1242


Failed to patch pandas - unable to change molecule rendering
Failed to patch pandas - unable to change molecule rendering


# smiles: 1740
# duplicates: 0
# discarded molecules: 0
# frames: 3482
# cuts:  4612
Time:  3.10151743888855
FPS:  1487.0140474376124
# frames: 5222
# cuts:  6352
# hcores:  5062
# cores in hcores:  5222
#hcores:  5062 #cpds:  6352 #unique cpds: 1740
Removed single-cpd series:
#hcores:  494 #cpds:  1784 #unique cpds: 1057
Removed sub-series:
#hcores:  304 #cpds:  1262 #unique cpds: 1057
Optimized R-sites
#hcores:  304 #cpds:  1262 #unique cpds: 1057
Unique assignments:
#hcores:  205 #cpds:  995 #unique cpds: 995
Optimized R-sites for unique ccrs
#hcores:  205 #cpds:  995 #unique cpds: 995


Failed to patch pandas - unable to change molecule rendering
Failed to patch pandas - unable to change molecule rendering


In [23]:
pd.read_csv('df_ccr_results_Q16790.csv')

,core,substituents,cid,tid,smiles,analogue_series_id
0,O=c1cc([*:3])c2cc([*:2])c(O[*:1])c([*:4])c2o1,C#CCCC[*:1].[*:2].[*:3].[*:4],CHEMBL3121782,CHEMBL3594,C#CCCCOc1ccc2ccc(=O)oc2c1,0
1,O=c1cc([*:3])c2cc([*:2])c(O[*:1])c([*:4])c2o1,C#CC[*:1].[*:2].[*:3].[*:4],CHEMBL308819,CHEMBL3594,C#CCOc1ccc2ccc(=O)oc2c1,0
2,O=c1cc([*:3])c2cc([*:2])c(O[*:1])c([*:4])c2o1,C=CC[*:1].[*:2].[*:3].[*:4],CHEMBL432745,CHEMBL3594,C=CCOc1ccc2ccc(=O)oc2c1,0
3,O=c1cc([*:3])c2cc([*:2])c(O[*:1])c([*:4])c2o1,CC(=O)[*:1].[*:2].[*:3].[*:4],CHEMBL487711,CHEMBL3594,CC(=O)Oc1ccc2ccc(=O)oc2c1,0
4,O=c1cc([*:3])c2cc([*:2])c(O[*:1])c([*:4])c2o1,CCCC[*:1].[*:2].[*:3].[*:4],CHEMBL571514,CHEMBL3594,CCCCOc1ccc2ccc(=O)oc2c1,0
...,...,...,...,...,...,...
1715,O=P(O)(O)C(Nc1cccc([*:1])c1)P(=O)(O)O,[*:1],CHEMBL2440580,CHEMBL3594,O=P(O)(O)C(Nc1ccccc1)P(=O)(O)O,430
1716,O=P(O)(O)C(Nc1cccc([*:1])c1)P(=O)(O)O,c1ccc([*:1])cc1,CHEMBL55371,CHEMBL3594,O=P(O)(O)C(Nc1cccc(-c2ccccc2)c1)P(=O)(O)O,430
1717,O=c1[nH]c2cc(Cl)c([*:1])cc2c(=O)n1O,[*:1],CHEMBL80594,CHEMBL3594,O=c1[nH]c2cc(Cl)ccc2c(=O)n1O,431
1718,O=c1[nH]c2cc(Cl)c([*:1])cc2c(=O)n1O,c1cn([*:1])cn1,CHEMBL4105134,CHEMBL3594,O=c1[nH]c2cc(Cl)c(-n3ccnc3)cc2c(=O)n1O,431
